# Flow Inference Trajectory Viewer

Use ASE's NGL notebook viewer for flow trajectories.

In [1]:
from pathlib import Path
import json

import numpy as np
from ase.io import read
from ase.visualize import view

SANITY_ROOT = Path('/home/irteam/runs/overlap_sanity_50step')
RUN_NAME = 'fixedscale'  # 'fixedscale' or 'statnorm'
traj_dir = Path('/home/irteam/runs/dissoc_traj/catflow_center_rel/trajectories')
entries = json.loads((traj_dir / '_index.json').read_text())
entries

[{'sample_index': 0,
  'record_index': 0,
  'sid': -1,
  'system_key': None,
  'config_key': None,
  'has_dissoc': True,
  'has_overlap': False,
  'has_desorbed': False,
  'has_intercalated': False,
  'has_surf_changed': False,
  'valid_strict': False,
  'min_pair_distance_A': 1.0748904755503923,
  'n_frames': 51,
  'traj': '/home/irteam/runs/dissoc_traj/catflow_center_rel/trajectories/sample_000_sid_-1_flow.traj',
  'xyz': '/home/irteam/runs/dissoc_traj/catflow_center_rel/trajectories/sample_000_sid_-1_flow.xyz'},
 {'sample_index': 1,
  'record_index': 2,
  'sid': -1,
  'system_key': None,
  'config_key': None,
  'has_dissoc': True,
  'has_overlap': False,
  'has_desorbed': False,
  'has_intercalated': False,
  'has_surf_changed': False,
  'valid_strict': False,
  'min_pair_distance_A': 0.9747995327693256,
  'n_frames': 51,
  'traj': '/home/irteam/runs/dissoc_traj/catflow_center_rel/trajectories/sample_001_sid_-1_flow.traj',
  'xyz': '/home/irteam/runs/dissoc_traj/catflow_center_rel/t

In [2]:
# Helper: pull canonical adsorbate geometry by record_index → LMDB → adsorbates.pkl
import pickle, lmdb

_ADS_DB = pickle.load(open('/home/irteam/data/pkls/adsorbates.pkl', 'rb'))
_OC20_ENV = lmdb.open(
    '/home/irteam/data/processed/oc20dense.lmdb',
    subdir=False, readonly=True, lock=False, readahead=False,
)

def get_ads_info(record_index: int) -> dict:
    """oc20dense entry[record_index] → ads_id → canonical Atoms from adsorbates.pkl."""
    with _OC20_ENV.begin() as txn:
        entry = pickle.loads(txn.get(str(int(record_index)).encode('ascii')))
    ads_id = int(entry.get('ads_id', -1))
    atoms, smiles, binding_idx, label = _ADS_DB[ads_id]
    return {
        'ads_id': ads_id,
        'smiles': smiles,
        'binding_idx': binding_idx,
        'canonical_atoms': atoms,
        'lmdb_entry': entry,
    }

def extract_predicted_ads(frame):
    """Keep only tag==2 atoms; drop cell/pbc so local geometry is viewable in isolation."""
    tags = frame.get_tags()
    ads_idx = [i for i, t in enumerate(tags) if t == 2]
    sub = frame[ads_idx]
    sub.set_pbc(False)
    sub.set_cell(None)
    return sub


## Anomaly-only filter (canonical-reference dissoc)

Recompute anomaly status using the canonical adsorbate bond graph instead of x_0's
bond graph (the x_0 graph is unreliable when the prior is a non-rigid Gaussian).
Then narrow `entries` to only samples that are still anomalous under that
stricter, correct judgement.


In [3]:
import numpy as np
from ase import Atoms as _Atoms
from ase.io import read as _read
from fairchem.data.oc.utils import DetectTrajAnomaly as _Detect

def canonical_dissoc(entry):
    """True if final ads bond graph differs from canonical adsorbate bond graph."""
    try:
        info = get_ads_info(entry['record_index'])
    except Exception:
        return None
    canonical = info['canonical_atoms']
    final = _read(entry['traj'], index=-1)
    z = np.asarray(final.get_atomic_numbers(), dtype=np.int64)
    tags = np.asarray(final.get_tags(), dtype=np.int64)
    ads_mask = (tags == 2)
    canon_z = np.asarray(canonical.get_atomic_numbers(), dtype=np.int64)
    if canon_z.size != int(ads_mask.sum()) or not np.array_equal(canon_z, z[ads_mask]):
        return None
    init_pos = final.get_positions().copy()
    init_pos[ads_mask] = canonical.get_positions()
    init_atoms = _Atoms(numbers=z, positions=init_pos, cell=final.cell, pbc=True)
    init_atoms.set_tags(tags.tolist())
    det = _Detect(init_atoms=init_atoms, final_atoms=final, atoms_tag=tags.tolist())
    return bool(det.is_adsorbate_dissociated())

def is_real_anomaly(entry):
    cd = canonical_dissoc(entry)
    return bool(
        (cd is True)
        or entry.get('has_overlap')
        or entry.get('has_intercalated')
        or entry.get('has_desorbed')
        or entry.get('has_surf_changed')
    )

# Annotate every entry with the canonical-reference dissoc flag for later use.
for e in entries:
    e['canon_dissoc'] = canonical_dissoc(e)

anomaly_entries = [e for e in entries if is_real_anomaly(e)]

print(f"all entries: {len(entries)}  |  canonical-anomaly only: {len(anomaly_entries)}")
print()
print("Summary by entry:")
for i, e in enumerate(entries):
    flags = []
    if e.get('canon_dissoc'): flags.append('DISSOC*')   # canonical-reference
    if e.get('has_overlap'): flags.append('OVERLAP')
    if e.get('has_intercalated'): flags.append('INTER')
    if e.get('has_desorbed'): flags.append('DESORB')
    if e.get('has_surf_changed'): flags.append('SURF')
    tag = '|'.join(flags) if flags else 'VALID'
    print(f"  entry {i:2d}  record={e['record_index']:3d}  ads_id={get_ads_info(e['record_index'])['ads_id']:3d}  [{tag}]  min_pair={e.get('min_pair_distance_A',float('nan')):.2f}A")


all entries: 20  |  canonical-anomaly only: 4

Summary by entry:
  entry  0  record=  0  ads_id= 20  [VALID]  min_pair=1.07A
  entry  1  record=  2  ads_id= 10  [VALID]  min_pair=0.97A
  entry  2  record=  3  ads_id=  8  [VALID]  min_pair=0.77A
  entry  3  record=  5  ads_id= 33  [VALID]  min_pair=1.07A
  entry  4  record=  7  ads_id= 25  [VALID]  min_pair=1.09A
  entry  5  record=  8  ads_id= 71  [INTER]  min_pair=0.99A
  entry  6  record= 10  ads_id= 71  [VALID]  min_pair=1.03A
  entry  7  record= 12  ads_id=  7  [VALID]  min_pair=1.16A
  entry  8  record= 13  ads_id=  8  [VALID]  min_pair=0.73A
  entry  9  record= 14  ads_id=  6  [VALID]  min_pair=1.06A
  entry 10  record= 15  ads_id=  8  [VALID]  min_pair=0.72A
  entry 11  record= 17  ads_id= 20  [VALID]  min_pair=1.09A
  entry 12  record= 18  ads_id= 41  [OVERLAP]  min_pair=0.44A
  entry 13  record= 19  ads_id=  8  [VALID]  min_pair=0.65A
  entry 14  record= 20  ads_id= 35  [VALID]  min_pair=1.03A
  entry 15  record= 21  ads_id= 5

In [4]:
# Toggle this to True to restrict downstream cells to anomaly-only samples.
USE_ANOMALY_ONLY = True

active_entries = anomaly_entries if USE_ANOMALY_ONLY else entries
print(f"active_entries: {len(active_entries)}")


active_entries: 4


In [8]:
ENTRY_INDEX = 1
entry = active_entries[ENTRY_INDEX]
frames = read(entry['traj'], index=':')

print(entry)
print(f"{len(frames)} frames, {len(frames[0])} atoms")
print(f"canon_dissoc={entry.get('canon_dissoc')}  has_overlap={entry.get('has_overlap')}  has_intercalated={entry.get('has_intercalated')}")
view(frames, viewer='ngl')


{'sample_index': 12, 'record_index': 18, 'sid': -1, 'system_key': None, 'config_key': None, 'has_dissoc': True, 'has_overlap': True, 'has_desorbed': False, 'has_intercalated': False, 'has_surf_changed': False, 'valid_strict': False, 'min_pair_distance_A': 0.4364779341153388, 'n_frames': 51, 'traj': '/home/irteam/runs/dissoc_traj/catflow_center_rel/trajectories/sample_012_sid_-1_flow.traj', 'xyz': '/home/irteam/runs/dissoc_traj/catflow_center_rel/trajectories/sample_012_sid_-1_flow.xyz', 'canon_dissoc': False}
51 frames, 67 atoms
canon_dissoc=False  has_overlap=True  has_intercalated=False


In [6]:
# Optional: inspect the final overlap pair distance over the Euler trajectory.
pair = entry.get('global_min_pair')
if pair is not None:
    ds = np.array([atoms.get_distance(pair[0], pair[1], mic=True) for atoms in frames])
    print('pair:', pair)
    print('min distance:', ds.min())
    print('final distance:', ds[-1])
    print('first frame below 0.5 A:', int(np.argmax(ds < 0.5)) if np.any(ds < 0.5) else None)
    ds

## Compare model-generated local geometry with canonical adsorbate

Pulls the canonical (gas-phase) adsorbate `Atoms` for the current `entry` from
`adsorbates.pkl` and compares it visually with the model's predicted ads atoms.


In [7]:
info = get_ads_info(entry['record_index'])
print(f"ads_id={info['ads_id']}  smiles={info['smiles']}  binding_idx={info['binding_idx']}")
print(f"has_dissoc={entry['has_dissoc']}  has_overlap={entry['has_overlap']}  n_frames={entry['n_frames']}")

# Canonical reference geometry (gas-phase) from adsorbates.pkl
view(info['canonical_atoms'], viewer='ngl')


ads_id=71  smiles=*NH2  binding_idx=[0]
has_dissoc=True  has_overlap=False  n_frames=51


In [ ]:
# Model's predicted ads atoms at the final Euler step (pbc/cell stripped)
pred_ads_final = extract_predicted_ads(frames[-1])
print('predicted ads atomic numbers:', pred_ads_final.get_atomic_numbers().tolist())
print('canonical ads atomic numbers:', info['canonical_atoms'].get_atomic_numbers().tolist())
view(pred_ads_final, viewer='ngl')


In [ ]:
# Full trajectory of ads-only motion (cell/pbc dropped)
pred_ads_traj = [extract_predicted_ads(f) for f in frames]
view(pred_ads_traj, viewer='ngl')
